In [4]:
import rasterio
import numpy as np
import pandas as pd
import os

In [5]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [29]:
def extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping):

    """
    Extract pixel values from TIFF files for each buoy in loc_boyas and for each specified date.
    """
    results = []
    target_dates = sorted(set(target_dates))
    tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]
    #print(tif_files)

    for date in target_dates:
        date_to_find = date.replace('-', '')
        matching = [f for f in tif_files if date_to_find in f]
        if matching:
            tiff_file = os.path.join(folder_path, matching[0])  

            with rasterio.open(tiff_file) as dataset:
                print(f"Processing {tiff_file}")
                bands = dataset.read()
                print(bands.shape[0])
                for _, row in loc_boyas.iterrows():
                    buoy_id = row["CodPuntoControl"].replace('-', '').strip()
                    lat = int(round(row["LatitudEPSG32630"]))
                    lon = int(round(row["LongitudEPSG32630"]))

                    try:
                        row_idx, col_idx = dataset.index(lon, lat)
                        #print(row_idx, col_idx)

                        if grouping == "3x3":
                            offset = 1
                        elif grouping == "5x5":
                            offset = 2
                        elif grouping == "9x9":
                            offset = 4
                        else:
                            offset = 0

                        if offset > 0:
                            window = (
                                slice(max(row_idx - offset, 0), min(row_idx + offset + 1, dataset.height)),
                                slice(max(col_idx - offset, 0), min(col_idx + offset + 1, dataset.width))
                            )
                            reflectances = bands[:, window[0], window[1]]
                            values = np.median(reflectances, axis=(1, 2))
                        else:
                            values = bands[:, row_idx, col_idx]

                        if values.shape == (8,):
                            band_names = {f"Band_{i+1}": val for i, val in enumerate(values)}
                        elif values.shape == (4,):
                            band_names = {f"Band_{i+2}": np.float64(val) for i, val in enumerate(values)}
                        else:
                            band_names = {f"Band_{i+1}": val for i, val in enumerate(values)} 

                        results.append({
                            "Date": date,
                            "Buoy": buoy_id,
                            "Latitude": lat,
                            "Longitude": lon,
                            **band_names
                        })

                    except IndexError:
                        print(f"Skipping {buoy_id} on {date}: Coordinates out of raster bounds")

    return pd.DataFrame(results)

In [30]:

folder_path = "Copernicus/planet/"
target_dates = [
    '2019-09-19', '2021-09-30', '2019-10-30'
]


groupings = ["1x1"]#, "3x3", "5x5", "9x9"]
for grouping in groupings:

    df_tiffs = extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping)
    df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])

    df_tiffs.to_csv(f"saved_files/df_tifs_planet_{grouping}.csv", index=False)


Processing Copernicus/planet/20190919_composite_10m.tif
4
Processing Copernicus/planet/20191030_composite_10m.tif
4
Processing Copernicus/planet/20210930_composite_10m.tif
8


In [32]:
df_tiffs

,Date,Buoy,Latitude,Longitude,Band_2,Band_3,Band_4,Band_5,Band_1,Band_6,Band_7,Band_8
0,2019-09-19,CTD1,4187246,695025,9513.0,8316.0,6896.0,4504.0,NaN,NaN,NaN,NaN
1,2019-09-19,CTD2,4181518,693105,4802.0,3896.0,2215.0,983.0,NaN,NaN,NaN,NaN
2,2019-09-19,CTD3,4181698,695238,4792.0,3690.0,2096.0,889.0,NaN,NaN,NaN,NaN
3,2019-09-19,CTD4,4180266,698264,5043.0,3832.0,2297.0,1009.0,NaN,NaN,NaN,NaN
4,2019-09-19,CTD5,4179450,700268,10427.0,9017.0,7416.0,5170.0,NaN,NaN,NaN,NaN
5,2019-09-19,CTD6,4176009,695829,4827.0,3690.0,2021.0,878.0,NaN,NaN,NaN,NaN
6,2019-09-19,CTD7,4176724,690397,4813.0,3704.0,2123.0,1113.0,NaN,NaN,NaN,NaN
7,2019-09-19,CTD8,4174178,693048,4646.0,3536.0,2049.0,914.0,NaN,NaN,NaN,NaN
8,2019-09-19,CTD9,4171106,693183,4808.0,3752.0,2245.0,1047.0,NaN,NaN,NaN,NaN
9,2019-09-19,CTD10,4170388,695646,4767.0,3521.0,2046.0,971.0,NaN,NaN,NaN,NaN
